# Validation of Predicted Span Filtering

This notebook examines whether short model-predicted spans should be removed before the analysis of NLPCSS-20. The filtering decision is evaluated against the manually annotated AllSides 45 gold spans and the definition of an argumentative discourse unit used in the annotation scheme.

The analysis first examines the length distribution of gold argumentative discourse units and then assesses the prevalence and label distribution of one-word predictions in NLPCSS-20.

In [1]:
import pandas as pd
from pathlib import Path
import json

## 1. Gold Span Length Distribution

The manually annotated AllSides 45 articles are used as a reference for the expected length of valid argumentative discourse units. The goal is to determine whether very short model predictions are also observed among manually annotated units.

In [2]:
GOLD_SPAN_FILE = Path(
    "../data/bio_modernbert_tokens/allsides_45_gold_merged.jsonl"
)

with GOLD_SPAN_FILE.open("r", encoding="utf-8") as f:
    gold_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

In [3]:
rows = []

for record in gold_records:
    article_id = record["id"]
    article_text = record["text"]

    for span in record["spans"]:
        span_text = article_text[span["start"]:span["end"]]

        rows.append({
            "article_id": article_id,
            "label": span["label"],
            "start": span["start"],
            "end": span["end"],
            "text": span_text,
            "span_word_count": len(span_text.split()),
        })

gold_spans_df = pd.DataFrame(rows)

print("Gold argumentative units:", len(gold_spans_df))
gold_spans_df.head()

Gold argumentative units: 1644


,article_id,label,start,end,text,span_word_count
0,article_01,assumption,0,108,Bernie Sanders on Thursday gave the strongest ...,19
1,article_01,testimony,110,250,telling reporters after meeting with President...,22
2,article_01,testimony,313,451,The Vermont senator declared that he will stay...,22
3,article_01,testimony,456,542,said he wants to see whether the final vote co...,16
4,article_01,assumption,544,633,He also is planning to go forward with a big r...,17


### Gold Span Length Statistics

The distribution of gold span lengths is examined to determine whether extremely short spans occur among manually annotated argumentative discourse units. Particular attention is given to spans of five words or fewer.

In [4]:
gold_spans_df["span_word_count"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    1644.000000
mean       21.480535
std        12.269404
min         2.000000
1%          3.000000
5%          6.000000
10%         8.000000
25%        13.000000
50%        20.000000
75%        28.000000
90%        37.000000
95%        43.850000
99%        57.000000
max       121.000000
Name: span_word_count, dtype: float64

In [5]:
for cutoff in [1, 2, 3, 4, 5]:
    n = (gold_spans_df["span_word_count"] <= cutoff).sum()
    pct = (gold_spans_df["span_word_count"] <= cutoff).mean() * 100

    print(
        f"{cutoff} words or fewer: "
        f"{n} spans ({pct:.2f}%)"
    )

1 words or fewer: 0 spans (0.00%)
2 words or fewer: 4 spans (0.24%)
3 words or fewer: 18 spans (1.09%)
4 words or fewer: 32 spans (1.95%)
5 words or fewer: 57 spans (3.47%)


In [7]:
gold_spans_df[
    gold_spans_df["span_word_count"] <= 5
][
    ["label", "span_word_count", "text"]
].sort_values(
    ["span_word_count", "label"]
)

,label,span_word_count,text
1591,anecdote,2,And murdering
531,assumption,2,Nobody knows
1372,assumption,2,It wasn't
1146,other,2,Got that?
1592,assumption,3,But mostly resisting
1601,assumption,3,It was insane
1604,assumption,3,motives are hazier
1619,assumption,3,"Words, not guns"
1630,assumption,3,They’ll be isolated
1643,assumption,3,They never do


## 2. One-Word Predictions in NLPCSS-20

The filtered NLPCSS-20 prediction file has already had punctuation-only and formatting-only spans removed. Here, I examine whether one-word predictions should also be excluded based on the gold-span analysis above.

In [8]:
PREDICTIONS_FILE = Path(
    "../results/model_predictions/nlpcss20_model_predictions_filtered.csv"
)

predictions_df = pd.read_csv(
    PREDICTIONS_FILE,
    usecols=["label", "confidence", "text"],
    keep_default_na=False,
)

predictions_df["span_word_count"] = (
    predictions_df["text"]
    .str.split()
    .str.len()
)

print("Total predictions:", len(predictions_df))
print(
    "One-word predictions:",
    (predictions_df["span_word_count"] == 1).sum(),
)

predictions_df.head()

Total predictions: 496501
One-word predictions: 93635


,label,confidence,text,span_word_count
0,assumption,0.402250,administration,1
1,assumption,0.382558,Fisk,1
2,assumption,0.473326,Republican strategist,2
3,assumption,0.468179,Turner,1
4,assumption,0.401703,in,1


### Distribution of One-Word Predictions by Label

The frequency of one-word predictions is examined separately for each argumentation-unit type. This shows whether the short-span problem affects some predicted labels more strongly than others.

Even when the one-word prediction is semantically related to a label. For example "364" as statistics or "According" as statistics, the span itself is not a complete argumentative unit.

In [9]:
labels = [
    "anecdote",
    "assumption",
    "testimony",
    "statistics",
    "other",
]

total_by_label = (
    predictions_df["label"]
    .value_counts()
    .reindex(labels)
)

one_word_by_label = (
    predictions_df.loc[
        predictions_df["span_word_count"] == 1,
        "label",
    ]
    .value_counts()
    .reindex(labels)
    .fillna(0)
    .astype(int)
)

one_word_summary = pd.DataFrame({
    "total_predictions": total_by_label,
    "one_word_predictions": one_word_by_label,
})

one_word_summary["one_word_percentage"] = (
    one_word_summary["one_word_predictions"]
    / one_word_summary["total_predictions"]
    * 100
)

one_word_summary.round(2)

,total_predictions,one_word_predictions,one_word_percentage
label,,,
anecdote,125275,31725,25.32
assumption,188560,31393,16.65
testimony,155930,21371,13.71
statistics,16643,3858,23.18
other,10093,5288,52.39


### Examples of One-Word Predictions

A small random sample of one-word predictions is inspected for each label to assess whether these spans form complete argumentative discourse units or instead reflect segmentation errors.

In [10]:
for label in labels:
    print(f"\n {label}")

    display(
        predictions_df[
            (predictions_df["label"] == label)
            & (predictions_df["span_word_count"] == 1)
        ][
            ["text", "confidence"]
        ]
        .sample(n=10, random_state=42)
    )


 anecdote


,text,confidence
332467,"Later,",0.603146
470925,sitting,0.571847
383357,regional,0.489699
110001,In,0.394523
68427,and,0.631833
232397,COLUMBUS,0.552368
267643,And,0.377734
22162,letter,0.549485
174965,At,0.560663
160681,arranging,0.402710



 assumption


,text,confidence
123662,"debt,",0.547930
476409,She,0.347201
436951,on,0.555269
179404,exploring,0.661728
343121,play?,0.693585
333929,to,0.503518
383416,Department,0.476924
196024,reasonable,0.409714
413714,enroll,0.474123
396309,The,0.495878



 testimony


,text,confidence
418415,his,0.344400
235191,the,0.755663
433835,tunnel,0.558538
169964,denials,0.529467
193787,The,0.824370
478046,was,0.541900
457646,Noorani,0.525096
122186,Embassy,0.393099
225474,In,0.498526
48184,The,0.973477



 statistics


,text,confidence
273562,spent,0.504056
199145,More,0.535122
375966,364,0.542771
342983,seventh,0.568897
471255,The,0.528317
381550,the,0.409604
141949,'s,0.418278
423063,a,0.471635
55264,as,0.865573
371407,According,0.827722



 other


,text,confidence
33708,Reporting,0.575339
108930,Reporting,0.508100
205474,McCain,0.515501
416869,Press,0.700227
485324,at,0.565436
69011,Morgan,0.376829
203752,Ted,0.415028
393581,who,0.420735
430193,s,0.508682
50204,etuckerAP,0.682582


### Effect of the One-Word Filter

Because no one-word argumentative discourse units occur in the manually annotated gold sample and the inspected one-word predictions are incomplete fragments, one-word predictions are treated as segmentation errors. Before applying the filter to the final corpus, its effect on the predicted label distribution is examined.

In [13]:
keep_mask = predictions_df["span_word_count"] > 1

before_counts = (
    predictions_df["label"]
    .value_counts()
    .reindex(labels)
)

after_counts = (
    predictions_df.loc[keep_mask, "label"]
    .value_counts()
    .reindex(labels)
)

comparison_df = pd.DataFrame({
    "before_count": before_counts,
    "after_count": after_counts,
})

comparison_df["before_percentage"] = (
    comparison_df["before_count"]
    / before_counts.sum()
    * 100
)

comparison_df["after_percentage"] = (
    comparison_df["after_count"]
    / after_counts.sum()
    * 100
)

comparison_df["percentage_point_change"] = (
    comparison_df["after_percentage"]
    - comparison_df["before_percentage"]
)

print("Spans before filtering:", len(predictions_df))
print("One-word spans removed:", (~keep_mask).sum())
print("Spans after filtering:", keep_mask.sum())

comparison_df.round(2)

Spans before filtering: 496501
One-word spans removed: 93635
Spans after filtering: 402866


,before_count,after_count,before_percentage,after_percentage,percentage_point_change
label,,,,,
anecdote,125275,93550,25.23,23.22,-2.01
assumption,188560,157167,37.98,39.01,1.03
testimony,155930,134559,31.41,33.40,1.99
statistics,16643,12785,3.35,3.17,-0.18
other,10093,4805,2.03,1.19,-0.84


## 3. Save Final Filtered Predictions

The final prediction file excludes punctuation-only or formatting-only spans from the earlier filtering stage and additionally removes one-word predictions. The original prediction columns are preserved for the corpus analysis.

In [14]:
OUTPUT_FILE = Path(
    "../results/model_predictions/nlpcss20_model_predictions_filtered_no_one_word.csv"
)

first_chunk = True
rows_saved = 0

for chunk in pd.read_csv(
    PREDICTIONS_FILE,
    keep_default_na=False,
    chunksize=50_000,
):
    span_word_count = (
        chunk["text"]
        .str.split()
        .str.len()
    )

    filtered_chunk = chunk[
        span_word_count > 1
    ]

    filtered_chunk.to_csv(
        OUTPUT_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
    )

    rows_saved += len(filtered_chunk)
    first_chunk = False

print("Saved:", OUTPUT_FILE)
print("Rows saved:", rows_saved)

Saved: ../results/model_predictions/nlpcss20_model_predictions_filtered_no_one_word.csv
Rows saved: 402866
